# Laboratorio 11 — El disco lleno

**Duración: 45 minutos.**

En el laboratorio 08 aprendiste a ordenar la libreta. Hoy vas a ver **cuánto cuesta no
hacerlo**, con el número del disco en pantalla.

La pregunta de hoy la hace siempre el que paga el almacenamiento:

> Si la tabla tiene las mismas treinta mil filas que ayer, **¿por qué ocupa el doble?**

---

### En el almacén

Cada corrección que hiciste en la libreta no borró la página anterior: escribió una nueva.
Las viejas siguen en el cajón, ocupando lugar, y nadie las va a botar hasta que alguien lo
diga.

Eso no es un defecto. Es lo que te permitió, en el laboratorio 01, leer cómo estaba la
tabla el martes pasado. Pero tiene un precio, y hoy lo vas a medir.


## Paso 0 — Un año de documentos

Primero, a tu espacio de trabajo.

In [ ]:
%%sql
-- Celda 0.1



Partimos de cero.

In [ ]:
%%sql
-- Celda 0.2



Un año completo de documentos, los mismos treinta mil del laboratorio 04.

In [ ]:
%%sql
-- Celda 0.3



## Paso 1 — Medir el disco

Ahora la primera medición, y hay **dos formas de preguntar** que no dan lo mismo.

La primera se la preguntas a la libreta: cuántos archivos de datos cuentan hoy y cuánto
pesan.

In [ ]:
%%sql
-- Celda 1.1



La segunda se la preguntas al **disco**, que no sabe nada de libretas y solo ve
archivos. Esta celda no lleva `%%sql`: es una orden del sistema, y por eso empieza con
`!`.

> **Esta celda viene escrita, y hay otras dos iguales más adelante.** Una orden del
> sistema no se puede dejar a medias como una sentencia SQL, así que las tres llegan
> completas y se ejecutan tal cual.
>
> Son las tres mediciones del disco, que es lo que venimos a ver hoy. Córrelas en orden y
> anota los números, porque la gracia está en compararlos.

In [ ]:
# Celda 1.2
!hdfs dfs -du -s /warehouse/iceberg/mi_espacio.db/documentos_lab11

**Anota los dos números.** No son iguales, y la diferencia tiene explicación: la
libreta cuenta solo los archivos de datos, y el disco cuenta además los manifiestos y los
`metadata.json`, que también ocupan.

Los vas a comparar tres veces hoy.

## Paso 2 — Cinco correcciones

Ahora cinco correcciones seguidas sobre la misma columna. Son cinco `UPDATE` que no
agregan ni quitan una sola fila: solo cambian el estado de algunos documentos.

In [ ]:
%%sql
-- Celda 2.1



La segunda, sobre febrero.

In [ ]:
%%sql
-- Celda 2.2



La tercera, sobre marzo.

In [ ]:
%%sql
-- Celda 2.3



La cuarta, sobre abril.

In [ ]:
%%sql
-- Celda 2.4



Y la quinta, sobre mayo.

In [ ]:
%%sql
-- Celda 2.5



Cinco correcciones hechas. Comprueba primero que la tabla tiene las mismas filas que
antes.

In [ ]:
%%sql
-- Celda 2.6



Los mismos treinta mil. Ahora vuelve a medir el disco.

In [ ]:
# Celda 2.7
!hdfs dfs -du -s /warehouse/iceberg/mi_espacio.db/documentos_lab11

## Paso 3 — Dónde se fue el espacio

**Creció, y bastante, con las mismas filas.** Ahí está la pregunta del laboratorio.

Para entender dónde se fue, mira la libreta. Seis páginas: la carga y las cinco
correcciones.

In [ ]:
%%sql
-- Celda 3.1



Y ahora la comparación que lo explica todo. `.files` son los archivos que **cuentan
hoy**; `.all_files` son **todos los que hay**, incluidos los de las páginas viejas que ya
nadie consulta pero que siguen en el disco.

In [ ]:
%%sql
-- Celda 3.2



## Paso 4 — Botar las páginas viejas

**Ahí está el espacio.** La diferencia entre las dos columnas es lo que ocupan las
páginas que ya nadie puede consultar sin pedirlas por su identificador.

Ahora las botamos, con el `expire_snapshots` del laboratorio 08. `retain_last => 1` deja
solo la página vigente.

**Ojo, esto no se deshace.** Después de esta celda, el `VERSION AS OF` de las páginas
viejas deja de funcionar. En producción, la fecha de corte es una decisión que se toma
antes, no después.

> **La fecha de esta celda es de laboratorio y no se copia a producción.** Dice 2030, una
> fecha que todavía no llega, y para el procedimiento eso significa *toda la historia es
> vieja*. Está puesta así para que hoy veas el efecto completo en una sola celda.
>
> En producción esa fecha es una decisión de retención que se toma **antes** de escribir la
> sentencia, y es la de la última compactación, como en el laboratorio 08. **Una fecha
> futura ahí se lleva toda la historia de la tabla**, y con ella cualquier viaje al pasado
> que alguien necesite mañana.

In [ ]:
%%sql
-- Celda 4.1



Vuelve a medir, por tercera y última vez.

In [ ]:
# Celda 4.2
!hdfs dfs -du -s /warehouse/iceberg/mi_espacio.db/documentos_lab11

**Bajó.** Y comprueba que los datos vigentes no se tocaron.

In [ ]:
%%sql
-- Celda 4.3



## Paso 5 — Los archivos que nadie reclama

Treinta mil, los mismos de siempre. Se botó la historia, no los datos.

Queda una última herramienta, y es distinta de la anterior. `expire_snapshots` borra lo
que la libreta declara como viejo. `remove_orphan_files` borra lo que **la libreta no
declara en absoluto**: archivos que quedaron en el directorio porque una escritura se
cortó a la mitad.

Y se corre como la viste en el laboratorio 08: con **`dry_run => true`**, que no borra
nada y solo lista lo que borraría. Se mira antes de borrar porque este procedimiento se
lleva **todo archivo que la tabla no reclame**, y un archivo que un trabajo está
escribiendo en este momento todavía no está declarado en ninguna página: para el
procedimiento es un huérfano. Primero se mira; borrar es el segundo paso, y solo con la
lista a la vista.

In [ ]:
%%sql
-- Celda 5.1



## Cierre

Los tres números que anotaste cuentan la historia completa:

| Momento | Qué pasó |
|---|---|
| Después de cargar | el tamaño de partida |
| Después de cinco correcciones | creció, con las mismas treinta mil filas |
| Después de expirar | volvió a bajar |

**La libreta guarda todo hasta que alguien le dice qué botar.** Eso no es un default que
haya que corregir: es la propiedad que te dejó viajar al pasado en el laboratorio 01.

Lo que sí es una decisión, y de las importantes, es **cuánta historia se conserva**. Se
llama política de retención, y hay que tomarla con dos preguntas sobre la mesa:

- **¿Cuánto hacia atrás necesita la Contraloría poder preguntar?** Eso fija el mínimo.
- **¿Cuánto disco hay?** Eso fija el máximo.

Si nadie la toma, el default es *guardar todo para siempre*, y el disco se llena un
viernes a las seis de la tarde.
